In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

your_key = x_api_key
url = "https://gateway.datacore.vn/data/ds/search"

headers = {
    "x-api-key": your_key,
    "Content-Type": "application/json"
}

dataset_code = "dataset_historical_price"
all_rows = []
fields = None
page = 1
limit = None   # đừng để 1000, vì backend có vẻ chỉ trả 100

while True:
    payload = {
        "dataSetCode": dataset_code,
        "conditions": [],
        "selectFields": [],
        "page": page,
        "limit": limit
    }

    res = requests.post(url, headers=headers, json=payload, timeout=60)
    res.raise_for_status()
    result = res.json()

    print(f"--- Page {page} ---")

    if "data" not in result:
        raise Exception(f"API error: {result}")

    data_block = result["data"]
    rows = data_block.get("dataDetail", [])

    if fields is None:
        fields = data_block.get("fields", [])

    print(f"Fetched {len(rows)} rows")

    # page hiện tại không còn data => dừng
    if not rows:
        print("No more rows, stop.")
        break

    all_rows.extend(rows)

    # thử đọc totalPages nếu backend có trả
    total_pages = data_block.get("totalPages")
    current_page = data_block.get("page", page)

    if total_pages is not None:
        print(f"totalPages = {total_pages}")
        if current_page >= total_pages:
            print("Reached last page.")
            break

    # không dùng len(rows) < limit để break nữa
    # vì backend có thể cap limit hoặc trả ít hơn limit nhưng vẫn còn page sau

    page += 1

df_stock_price = pd.DataFrame(all_rows, columns=fields)
print(df_stock_price.shape)
display(df_stock_price.head())

In [ ]:
import requests
import pandas as pd

url = "https://gateway.datacore.vn/data/ds/search"

headers = {
    "x-api-key": "REDACTED_LEAKED_API_KEY",
    "Content-Type": "application/json"
}

payload = {
    "dataSetCode": "fundamental_annual",
}

response = requests.post(url, headers=headers, json=payload)

data = response.json()
payload_data = data.get("data", {})

print("Status:", response.status_code)
print("Response:")
print({
    "num": payload_data.get("num"),
    "totalPage": payload_data.get("totalPage"),
    "currentPage": payload_data.get("currentPage")
})

if "data" in data:
    df = pd.DataFrame(payload_data["dataDetail"], columns=payload_data["fields"])
    print("\nDataFrame:")
    print(df.head())
else:
    print("\nKhông có data để convert DataFrame")

In [ ]:
import requests
import pandas as pd
your_key = "REDACTED_LEAKED_API_KEY"
url = "https://gateway.datacore.vn/data/ds/search"

headers = {
    "x-api-key": your_key,
    "Content-Type": "application/json"
}

payload = {
    "dataSetCode": "company_information",
    "conditions": [],
    "selectFields": [],
    "page": 1,
    "limit": 10}
res = requests.post(url, headers=headers, json=payload)
data = res.json()

df_company_infor = pd.DataFrame(
    data["data"]["dataDetail"],
    columns=data["data"]["fields"]
)

display(df_company_infor.head())

In [ ]:
from __future__ import annotations

import json
import time
from functools import wraps
from typing import Optional, Dict, Any, List, Iterator

import pandas as pd
import requests
from dotenv import dotenv_values

_env = dotenv_values()


class DatacoreError(Exception):
    """Base exception for Datacore package."""
    pass


class AuthenticationError(DatacoreError):
    """Raised when API key is missing or invalid."""
    pass


class PermissionDeniedError(DatacoreError):
    """Raised when access is denied."""
    pass


class APIRequestError(DatacoreError):
    """Raised when request fails or response format is invalid."""
    pass


def retry_on_error(max_retries: int = 3, delay: float = 1.0):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(max_retries):
                try:
                    return func(*args, **kwargs)
                except requests.RequestException as exc:
                    last_error = exc
                    if attempt < max_retries - 1:
                        time.sleep(delay * (2 ** attempt))
            raise last_error
        return wrapper
    return decorator


class Datacore:
    """
    Datacore API client.

    Demo:
        client = Datacore()
        df = client.preview("vsic")

    Paid:
        client = Datacore(api_key="your-api-key")
        raw = client.get_data("dataset_historical_price")
        df = client.get_dataframe("dataset_historical_price")
        info = client.get_data_info("dataset_historical_price")
    """

    GATEWAY_URL = _env.get("DATACORE_GATEWAY_URL", "https://gateway.datacore.vn")
    SEARCH_URL = _env.get("DATACORE_SEARCH", f"{GATEWAY_URL}/data/ds/search")
    PREVIEW_URL = _env.get("DATACORE_PREVIEW", "").strip()

    def __init__(self, api_key: Optional[str] = None, timeout: int = 30):
        self.timeout = timeout
        self.api_key = (api_key or _env.get("X_API_KEY") or "").strip()

        self.session = requests.Session()
        self.session.headers.update({
            "Content-Type": "application/json"
        })

        if self.api_key:
            self.session.headers.update({
                "x-api-key": self.api_key
            })

    def set_api_key(self, api_key: str) -> None:
        if not api_key or not api_key.strip():
            raise ValueError("API key cannot be empty.")
        self.api_key = api_key.strip()
        self.session.headers.update({
            "x-api-key": self.api_key
        })

    def is_authenticated(self) -> bool:
        return bool(self.api_key)

    def _require_auth(self) -> None:
        if not self.is_authenticated():
            raise AuthenticationError(
                "API key not found. Pass api_key to Datacore(api_key='your-key') "
                "or set X_API_KEY in .env"
            )

    def _raise_api_error(self, response: requests.Response) -> None:
        try:
            err_data = response.json()
        except ValueError:
            err_data = {}

        message = (
            err_data.get("message")
            or err_data.get("error")
            or err_data.get("detail")
            or response.text
            or f"HTTP {response.status_code}"
        )

        if response.status_code in (401, 403):
            lowered = str(message).lower()
            if any(word in lowered for word in ["permission", "forbidden", "denied", "not allowed"]):
                raise PermissionDeniedError(message)
            raise AuthenticationError(message)

        raise APIRequestError(message)

    @staticmethod
    def _get_payload(data: Dict[str, Any], error_prefix: str = "response") -> Dict[str, Any]:
        if not isinstance(data, dict):
            raise APIRequestError(
                f"Cannot parse {error_prefix}: response is not a dict"
            )

        payload = data.get("data")
        if payload is None:
            message = (
                data.get("message")
                or data.get("error")
                or data.get("detail")
                or "missing key 'data'"
            )
            raise APIRequestError(
                f"Cannot parse {error_prefix}: {message}"
            )

        if not isinstance(payload, dict):
            raise APIRequestError(
                f"Cannot parse {error_prefix}: key 'data' is not an object"
            )

        return payload

    @staticmethod
    def _extract_metadata(data: Dict[str, Any], queried_rows: Optional[int] = None) -> Dict[str, Any]:
        if not isinstance(data, dict):
            return {
                "num": None,
                "totalPage": None,
                "currentPage": None,
                "queried_rows": queried_rows,
            }

        payload = data.get("data", {})
        if not isinstance(payload, dict):
            payload = {}

        return {
            "num": data.get("num", payload.get("num")),
            "totalPage": data.get("totalPage", payload.get("totalPage")),
            "currentPage": data.get("currentPage", payload.get("currentPage")),
            "queried_rows": queried_rows,
        }

    @staticmethod
    def _metadata_to_string(meta: Dict[str, Any]) -> str:
        return (
            f"num: {meta.get('num')}, "
            f"totalPage: {meta.get('totalPage')}, "
            f"currentPage: {meta.get('currentPage')}, "
            f"queried_rows: {meta.get('queried_rows')}"
        )

    @staticmethod
    def _to_dataframe(data: Dict[str, Any], error_prefix: str = "response") -> pd.DataFrame:
        payload = Datacore._get_payload(data, error_prefix=error_prefix)

        rows = payload.get("dataDetail")
        fields = payload.get("fields")

        if rows is None:
            raise APIRequestError(
                f"Cannot convert {error_prefix} to DataFrame: missing key 'dataDetail'"
            )

        if fields is None:
            raise APIRequestError(
                f"Cannot convert {error_prefix} to DataFrame: missing key 'fields'"
            )

        try:
            return pd.DataFrame(rows, columns=fields)
        except Exception as exc:
            raise APIRequestError(
                f"Cannot convert {error_prefix} to DataFrame: {exc}"
            ) from exc

    @retry_on_error(max_retries=3, delay=1.0)
    def preview_raw(self, dataset_code: str) -> Dict[str, Any]:
        """
        Demo mode, no API key required.
        Need DATACORE_PREVIEW in .env
        """
        if not self.PREVIEW_URL:
            raise APIRequestError("DATACORE_PREVIEW is not configured in .env")

        response = requests.get(
            self.PREVIEW_URL,
            params={"dataSetCode": dataset_code},
            timeout=self.timeout,
        )

        if not response.ok:
            self._raise_api_error(response)

        return response.json()

    def preview(self, dataset_code: str) -> pd.DataFrame:
        data = self.preview_raw(dataset_code)
        return self._to_dataframe(data, error_prefix="preview response")

    @retry_on_error(max_retries=3, delay=1.0)
    def _request_data(
        self,
        dataset_code: str,
        conditions: Optional[List[Dict[str, Any]]] = None,
        select_fields: Optional[List[str]] = None,
        page: int = 1,
        limit: Optional[int] = None,
        **kwargs: Any,
    ) -> Dict[str, Any]:
        self._require_auth()

        payload = {
            "dataSetCode": dataset_code,
            "conditions": conditions or [],
            "selectFields": select_fields or [],
            "page": page,
            **kwargs,
        }

        if limit is not None:
            payload["limit"] = limit

        response = self.session.post(
            self.SEARCH_URL,
            json=payload,
            timeout=self.timeout,
        )

        if not response.ok:
            self._raise_api_error(response)

        return response.json()

    def get_data(
        self,
        dataset_code: str,
        conditions: Optional[List[Dict[str, Any]]] = None,
        select_fields: Optional[List[str]] = None,
        page: int = 1,
        limit: Optional[int] = None,
        return_type: str = "dict",
        **kwargs: Any,
    ):
        """
        return_type:
        - "dict": trả raw response dict
        - "json": trả JSON string
        - "dataframe": trả pandas DataFrame
        """
        raw_data = self._request_data(
            dataset_code=dataset_code,
            conditions=conditions,
            select_fields=select_fields,
            page=page,
            limit=limit,
            **kwargs,
        )

        if return_type == "dict":
            return raw_data

        if return_type == "json":
            return json.dumps(raw_data, indent=2, ensure_ascii=False)

        if return_type == "dataframe":
            return self._to_dataframe(raw_data, error_prefix="API response")

        raise ValueError("return_type must be 'dict', 'json', or 'dataframe'")

    def get_dataframe(
        self,
        dataset_code: str,
        conditions: Optional[List[Dict[str, Any]]] = None,
        select_fields: Optional[List[str]] = None,
        page: int = 1,
        limit: Optional[int] = None,
        **kwargs: Any,
    ) -> pd.DataFrame:
        raw_data = self.get_data(
            dataset_code=dataset_code,
            conditions=conditions,
            select_fields=select_fields,
            page=page,
            limit=limit,
            return_type="dict",
            **kwargs,
        )
        return self._to_dataframe(raw_data, error_prefix="API response")

    def get_data_info(
        self,
        dataset_code: str,
        conditions: Optional[List[Dict[str, Any]]] = None,
        select_fields: Optional[List[str]] = None,
        page: int = 1,
        limit: Optional[int] = None,
        **kwargs: Any,
    ) -> str:
        raw_data = self.get_data(
            dataset_code=dataset_code,
            conditions=conditions,
            select_fields=select_fields,
            page=page,
            limit=limit,
            return_type="dict",
            **kwargs,
        )

        try:
            df = self._to_dataframe(raw_data, error_prefix="API response")
            queried_rows = len(df)
        except Exception:
            queried_rows = None

        meta = self._extract_metadata(raw_data, queried_rows=queried_rows)
        return self._metadata_to_string(meta)

    def paginate(
        self,
        dataset_code: str,
        max_pages: Optional[int] = None,
        limit: Optional[int] = None,
        conditions: Optional[List[Dict[str, Any]]] = None,
        select_fields: Optional[List[str]] = None,
        **kwargs: Any,
    ) -> Iterator[pd.DataFrame]:
        """
        Paid API pagination.
        """
        page = 1
        while True:
            if max_pages is not None and page > max_pages:
                break

            df = self.get_dataframe(
                dataset_code=dataset_code,
                conditions=conditions,
                select_fields=select_fields,
                page=page,
                limit=limit,
                **kwargs,
            )

            if df.empty:
                break

            yield df
            page += 1